<a href="https://colab.research.google.com/github/zhangtaira71/house-prices/blob/main/Voting_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install catboost -q

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import VotingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd

url = "https://raw.githubusercontent.com/zhangtaira71/house-prices/main/train.csv"

train = pd.read_csv(url)

train.head()

print(train.shape)

train["TotalArea"] = (
    train["GrLivArea"]
    + train["TotalBsmtSF"]
)

train["HouseAge"] = (
    2026 - train["YearBuilt"]
)

train["TotalBath"] = (
    train["FullBath"]
    + train["HalfBath"] * 0.5
)

features = [
    'OverallQual',
    'GrLivArea',
    'GarageCars',
    'TotalBath',
    'GarageArea',
    'TotalArea',
    'HouseAge',
    'LotArea',
    'OverallCond',
    'TotRmsAbvGrd',
    'Fireplaces'
]

missing = (
    train[features]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

print("Missing Values:")
print(missing)

X = train[features]
y = train["SalePrice"]

from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# =====================
# Random Forest
# =====================
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)
rf_preds = rf.predict(X_valid)

rf_mae = mean_absolute_error(y_valid, rf_preds)

print("Random Forest MAE:", rf_mae)

# =====================
# XGBoost
# =====================
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_valid)

xgb_mae = mean_absolute_error(y_valid, xgb_preds)

print("XGBoost MAE:", xgb_mae)

# =====================
# CatBoost
# =====================
cat = CatBoostRegressor(
    iterations=500,
    depth=8,
    verbose=0,
    random_state=42
)

cat.fit(X_train, y_train)
cat_preds = cat.predict(X_valid)

cat_mae = mean_absolute_error(y_valid, cat_preds)

print("CatBoost MAE:", cat_mae)

# =====================
# Voting Ensemble
# =====================
voting_model = VotingRegressor([
    ('rf', rf),
    ('xgb', xgb),
    ('cat', cat)
])

voting_model.fit(X_train, y_train)

voting_preds = voting_model.predict(X_valid)

voting_mae = mean_absolute_error(
    y_valid,
    voting_preds
)

print("Voting Ensemble MAE:", voting_mae)



(1460, 81)
Missing Values:
OverallQual     0
GrLivArea       0
GarageCars      0
TotalBath       0
GarageArea      0
TotalArea       0
HouseAge        0
LotArea         0
OverallCond     0
TotRmsAbvGrd    0
Fireplaces      0
dtype: int64
Random Forest MAE: 17329.60136442705
XGBoost MAE: 17525.736328125
CatBoost MAE: 16977.232811535192
Voting Ensemble MAE: 16660.37820244779
